<a href="https://colab.research.google.com/github/zhiyuguo-innis/MSSP607/blob/main/WeeklyModules/Week10/PARTICIPATION_ACTIVITY_Week_10_Improving_Coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Participation Activity Week 10: Improving Coding
**MSSP 607** · Zhiyu Guo

Improved version of my Week 4 notebook *Opening a CSV file and modifying the contents*. The original code was reviewed with a generative AI tool (Claude), and its recommendations are applied below. The questions (a–f) and the data are unchanged.

Data: `parsed.csv` (USGS earthquake data). Source: Molin, S. (2019). *Hands-On Data Analysis with Pandas*, Ch. 2.

## AI recommendations and what changed

| # | Issue in the Week 4 code | Change made here |
|---|---|---|
| 1 | The answers were typed into markdown cells by hand (e.g. "**4.9**", "**23.13%**"). If the data or code changed, the text would silently disagree with the output. | Every answer is printed from the computed value with an f-string, and all answers are collected in one summary table at the end. |
| 2 | The same filtering pattern (`df[df["parsed_place"] == ...]`) was rewritten for each question. | A small helper, `quakes_in(place)`, does the filtering, so each question is one readable line. |
| 3 | The notebook only worked if GitHub was reachable. | It reads from GitHub and falls back to a local copy of `parsed.csv`. |
| 4 | No checks on the columns the answers depend on. | `assert` checks confirm `tsunami` holds only 0/1 and the needed columns exist. One missing `mag` value is reported rather than hidden. |
| 5 | `tsunami` was treated as a number in (b) (`.sum()`) and compared to 1 in (f). | `tsunami` is converted to True/False once, so `.sum()` counts tsunamis and `.mean()` gives the share directly. |
| 6 | `nevada.describe()` printed summary statistics for all 27 columns, mostly empty or meaningless (IDs, timestamps). | The summary is limited to the columns that describe the earthquakes (magnitude, significance `sig`, felt reports). |
| 7 | The Ring of Fire list was not checked against the data, so a misspelled place would match zero rows without any warning. | The notebook counts matches for every place in the list and stops if any place matches nothing. |
| 8 | Part (e) showed `True`/`False` counts with no labels or shares. | Counts are labeled ("Inside" / "Outside") and shown with percentages. |
| 9 | The CSV was opened and modified, but the modified version was never saved. | The data with the new `ring_of_fire` column are written to `parsed_with_ring_of_fire.csv`. |

The Ring of Fire place list itself is kept exactly as the textbook exercise defines it.

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

DATA_URL = ("https://raw.githubusercontent.com/zhiyuguo-innis/mssp607/main/"
            "data/parsed.csv")
LOCAL_COPIES = [Path("parsed.csv"), Path("data/parsed.csv")]   # used if GitHub cannot be reached
OUTPUT_FILE = Path("parsed_with_ring_of_fire.csv")

NEEDED_COLUMNS = ["mag", "magType", "tsunami", "parsed_place", "sig", "felt", "cdi"]

## 2. Load the data

In [2]:
def load_earthquakes():
    """Read parsed.csv from GitHub, or from a local copy if GitHub fails."""
    try:
        return pd.read_csv(DATA_URL), DATA_URL
    except Exception as err:
        print(f"GitHub read failed ({type(err).__name__}); trying a local copy.")
        for path in LOCAL_COPIES:
            if path.exists():
                return pd.read_csv(path), str(path)
        raise FileNotFoundError("parsed.csv not found on GitHub or locally.")

df, source = load_earthquakes()
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns from {source}")
df[["parsed_place", "mag", "magType", "tsunami"]].head()

Loaded 9,332 rows x 27 columns from https://raw.githubusercontent.com/zhiyuguo-innis/mssp607/main/data/parsed.csv


,parsed_place,mag,magType,tsunami
0,California,1.35,ml,0
1,California,1.29,ml,0
2,California,3.42,ml,0
3,California,0.44,ml,0
4,California,2.16,md,0


## 3. Check the data

In [3]:
missing_cols = [c for c in NEEDED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
assert set(df["tsunami"].unique()) <= {0, 1}, "tsunami should contain only 0 and 1"

df["tsunami"] = df["tsunami"].astype(bool)     # True = earthquake was coupled with a tsunami

print("Missing magnitude values:", df["mag"].isna().sum())
print("Missing place values:    ", df["parsed_place"].isna().sum())

Missing magnitude values: 1
Missing place values:     0


## 4. Helper function
Every question filters earthquakes by place, so the filter is written once.

In [4]:
def quakes_in(place, data=None):
    """Return the earthquakes whose parsed_place exactly equals `place`."""
    data = df if data is None else data
    return data[data["parsed_place"] == place]


answers = {}   # collects every answer for the summary table at the end

## a. 95th percentile of earthquake magnitude in Japan (magType `mb`)

In [5]:
japan_mb = quakes_in("Japan").query("magType == 'mb'")
p95 = japan_mb["mag"].quantile(0.95)

answers["a. 95th percentile magnitude, Japan (mb)"] = f"{p95:.1f}"
print(f"Among the {len(japan_mb)} mb-type earthquakes in Japan, "
      f"the 95th percentile magnitude is {p95:.1f}.")

Among the 50 mb-type earthquakes in Japan, the 95th percentile magnitude is 4.9.


## b. Percentage of earthquakes in Indonesia coupled with tsunamis

In [6]:
indonesia = quakes_in("Indonesia")
n_tsunami = indonesia["tsunami"].sum()
share = indonesia["tsunami"].mean() * 100     # mean of True/False = share that are True

answers["b. Indonesia earthquakes with tsunamis"] = f"{n_tsunami} of {len(indonesia)} ({share:.2f}%)"
print(f"{n_tsunami} of {len(indonesia)} earthquakes in Indonesia "
      f"({share:.2f}%) were coupled with tsunamis.")

34 of 147 earthquakes in Indonesia (23.13%) were coupled with tsunamis.


## c. Summary statistics for earthquakes in Nevada
Only the columns that describe the earthquakes themselves are summarized.

In [7]:
nevada = quakes_in("Nevada")
nevada[["mag", "sig", "felt", "cdi"]].describe().round(2)

,mag,sig,felt,cdi
count,681.0,681.00,15.00,15.00
mean,0.5,10.97,2.40,2.44
std,0.7,19.61,4.63,0.50
min,-0.5,0.00,1.00,2.00
25%,-0.1,0.00,1.00,2.00
50%,0.4,2.00,1.00,2.20
75%,0.9,12.00,1.00,2.90
max,2.9,129.00,19.00,3.30


In [8]:
mag = nevada["mag"]
answers["c. Nevada earthquakes (count, mean / median magnitude)"] = (
    f"{len(nevada)} quakes, mean {mag.mean():.2f}, median {mag.median():.1f}")

print(f"Nevada had {len(nevada)} earthquakes. Magnitudes were very small: "
      f"mean {mag.mean():.2f}, median {mag.median():.1f}, range {mag.min():.1f} to {mag.max():.1f}.")
print(f"Tsunamis: {nevada['tsunami'].sum()} (Nevada is landlocked). "
      f"Events with felt reports: {nevada['felt'].notna().sum()}.")

Nevada had 681 earthquakes. Magnitudes were very small: mean 0.50, median 0.4, range -0.5 to 2.9.
Tsunamis: 0 (Nevada is landlocked). Events with felt reports: 15.


## d. Ring of Fire indicator column
- **Exact matching** (`isin`) is used, so `New Mexico` is **not** selected when matching `Mexico`.
- **Antarctica** appears in the data as ocean ridges (e.g. `Pacific-Antarctic Ridge`), so it is matched by the substring `"Antarctic"`.

In [9]:
RING_OF_FIRE_PLACES = [
    "Bolivia", "Chile", "Ecuador", "Peru", "Costa Rica", "Guatemala", "Mexico",
    "Japan", "Philippines", "Indonesia", "New Zealand", "Canada", "Fiji",
    "Alaska", "Washington", "California", "Russia", "Taiwan", "Tonga",
    "Kermadec Islands",
]

# Check every place in the list really appears in the data (catches typos)
place_counts = df["parsed_place"].value_counts()
matches = pd.Series({p: place_counts.get(p, 0) for p in RING_OF_FIRE_PLACES}, name="earthquakes")
unmatched = matches[matches == 0].index.tolist()
assert not unmatched, f"These places match no rows (check spelling): {unmatched}"

is_antarctic = df["parsed_place"].str.contains("Antarctic", na=False)
df["ring_of_fire"] = df["parsed_place"].isin(RING_OF_FIRE_PLACES) | is_antarctic

print("New Mexico flagged?", df.loc[df["parsed_place"] == "New Mexico", "ring_of_fire"].any())
print("Antarctic rows flagged:", int(is_antarctic.sum()))
matches.sort_values(ascending=False).to_frame()

New Mexico flagged? False
Antarctic rows flagged: 4


,earthquakes
Alaska,3665
California,2861
Washington,157
Indonesia,147
Fiji,61
Japan,57
Canada,55
Mexico,38
Russia,33
Chile,31


## e. Number of earthquakes inside vs. outside the Ring of Fire

In [10]:
ring_counts = (df["ring_of_fire"]
               .map({True: "Inside", False: "Outside"})
               .value_counts()
               .to_frame("earthquakes"))
ring_counts["percent"] = (ring_counts["earthquakes"] / len(df) * 100).round(1)

answers["e. Earthquakes inside / outside the Ring of Fire"] = (
    f"{ring_counts.loc['Inside', 'earthquakes']:,} / {ring_counts.loc['Outside', 'earthquakes']:,}")
ring_counts

,earthquakes,percent
ring_of_fire,,
Inside,7188,77.0
Outside,2144,23.0


## f. Tsunami count along the Ring of Fire

In [11]:
ring_tsunamis = df.loc[df["ring_of_fire"], "tsunami"].sum()
all_tsunamis = df["tsunami"].sum()

answers["f. Tsunamis along the Ring of Fire"] = f"{ring_tsunamis} of {all_tsunamis} tsunamis"
print(f"{ring_tsunamis} earthquakes along the Ring of Fire were coupled with tsunamis "
      f"({ring_tsunamis / all_tsunamis:.0%} of all {all_tsunamis} tsunamis in the data).")

45 earthquakes along the Ring of Fire were coupled with tsunamis (74% of all 61 tsunamis in the data).


## 5. Save the modified data

In [12]:
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {OUTPUT_FILE} ({len(df):,} rows, now with a ring_of_fire column)")

try:                                   # download from Colab; skipped elsewhere
    from google.colab import files
    files.download(str(OUTPUT_FILE))
except ImportError:
    pass

Saved parsed_with_ring_of_fire.csv (9,332 rows, now with a ring_of_fire column)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Summary of answers
Generated from the computed values, so it always matches the code above.

In [13]:
pd.Series(answers, name="Answer").to_frame()

,Answer
"a. 95th percentile magnitude, Japan (mb)",4.9
b. Indonesia earthquakes with tsunamis,34 of 147 (23.13%)
"c. Nevada earthquakes (count, mean / median magnitude)","681 quakes, mean 0.50, median 0.4"
e. Earthquakes inside / outside the Ring of Fire,"7,188 / 2,144"
f. Tsunamis along the Ring of Fire,45 of 61 tsunamis
